# Classification and regression as imputation

Use missing target columns to run supervised classification and regression through the same imputation API.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.datasets import make_classification, make_regression
from sklearn.metrics import accuracy_score, f1_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src" / "mimic").exists() else Path.cwd().parent
SRC_DIR = PROJECT_ROOT / "src"
if SRC_DIR.exists() and str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from mimic import MIMIC, RandomForestPathEncoder, MixedFeatureDecoder

Xc, yc = make_classification(n_samples=180, n_features=5, n_informative=4, n_redundant=0, random_state=1)
Xr, yr = make_regression(n_samples=180, n_features=1, noise=8, random_state=1)
df = pd.DataFrame(Xc, columns=[f"x{i}" for i in range(5)])
df["reg_target"] = yr
df["class_target"] = np.where(yc == 1, "positive", "negative")
train_idx, test_idx = train_test_split(df.index, test_size=0.25, random_state=1, stratify=df["class_target"])
work = df.copy()
work.loc[test_idx, ["reg_target", "class_target"]] = np.nan
work.head()

,x0,x1,x2,x3,x4,reg_target,class_target
0,-1.268537,2.146588,0.974099,1.873720,-0.721623,9.079952,negative
1,-0.740851,-2.815701,-0.960704,-2.054678,-0.709752,38.794874,positive
2,1.181783,0.582278,-1.817317,0.288462,1.856460,27.098935,positive
3,2.164446,2.377071,0.421407,0.813605,-3.944326,NaN,NaN
4,-0.674519,0.515429,2.007468,-0.105680,-0.428581,17.151661,negative


In [2]:
mimic = MIMIC(
    regression_columns=["x0", "x1", "x2", "x3", "x4", "reg_target"],
    classification_columns=["class_target"],
    encoder=RandomForestPathEncoder(n_estimators=20, embedding_dim=8, random_state=1),
    decoder=MixedFeatureDecoder.random_forest(n_estimators=20, random_state=1),
    n_bootstrap=2,
    random_state=1,
)
mimic.fit(work)
pred = mimic.impute(work, columns=["reg_target", "class_target"])
y_reg_true = df.loc[test_idx, "reg_target"]
y_reg_pred = pred.loc[test_idx, "reg_target"].astype(float)
y_cls_true = df.loc[test_idx, "class_target"]
y_cls_pred = pred.loc[test_idx, "class_target"]
metrics = {
    "regression_rmse": np.sqrt(mean_squared_error(y_reg_true, y_reg_pred)),
    "regression_mae": mean_absolute_error(y_reg_true, y_reg_pred),
    "classification_accuracy": accuracy_score(y_cls_true, y_cls_pred),
    "classification_f1": f1_score(y_cls_true, y_cls_pred, pos_label="positive"),
}
metrics

{'regression_rmse': np.float64(48.65681703596354),
 'regression_mae': 40.00595277618694,
 'classification_accuracy': 0.7777777777777778,
 'classification_f1': 0.782608695652174}

In [3]:
mimic.confidence(work.loc[test_idx], columns=["reg_target", "class_target"]).head()

,row_index,column,task,prediction,observed,bias,variance,residual,uncertainty,discrepancy,...,probability_std,probability_min,probability_max,probability_margin,vote_counts,vote_fraction,disagreement,top_class,second_class,noise
0,166,reg_target,regression,28.440896,NaN,-4.290986,23.038713,NaN,4568.795965,NaN,...,NaN,NaN,NaN,NaN,None,NaN,NaN,None,None,4545.757252
1,149,reg_target,regression,-2.850429,NaN,-4.290986,959.871500,NaN,5505.628752,NaN,...,NaN,NaN,NaN,NaN,None,NaN,NaN,None,None,4545.757252
2,6,reg_target,regression,15.336898,NaN,-4.290986,645.045644,NaN,5190.802896,NaN,...,NaN,NaN,NaN,NaN,None,NaN,NaN,None,None,4545.757252
3,143,reg_target,regression,37.913798,NaN,-4.290986,691.952518,NaN,5237.709771,NaN,...,NaN,NaN,NaN,NaN,None,NaN,NaN,None,None,4545.757252
4,12,reg_target,regression,2.139716,NaN,-4.290986,437.988738,NaN,4983.745990,NaN,...,NaN,NaN,NaN,NaN,None,NaN,NaN,None,None,4545.757252
